# Semana 10 — Projeto: Fine-Tuning de um Encoder com Training Loop Manual
**Fase 2: Deep Learning e o Ecossistema Hugging Face** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/02-Phase-2-Deep-Learning/notebooks/semana_10_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-10-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/02-Phase-2-Deep-Learning/docs/Semana-10-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_10_gabarito.ipynb`.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
!pip -q install datasets

In [ ]:
import torch, time, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0); print(device)
MODELO = "distilbert-base-uncased"

## Passo 1 — Dados
AG News: manchetes em 4 classes. Usamos uma amostra pequena para treinar em poucos minutos na T4.

In [ ]:
ds = load_dataset("fancyzhx/ag_news")
classes = ds["train"].features["label"].names
treino = ds["train"].shuffle(seed=0).select(range(4000))
teste = ds["test"].shuffle(seed=0).select(range(1000))
print(classes, "\n", treino[0])

## Passo 2 — Tokenization e dynamic padding
Escreva a `collate_fn`: recebe uma lista de exemplos e devolve um dicionário de tensores com `input_ids`, `attention_mask` e `labels`. Use `padding=True` (preenche até o maior do batch) e `truncation=True, max_length=128`.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)

def collate_fn(exemplos):
    # [SEU CÓDIGO AQUI] tokenize os textos do batch e adicione "labels"

dl_tr = DataLoader(treino, batch_size=32, shuffle=True, collate_fn=collate_fn)
dl_te = DataLoader(teste, batch_size=64, collate_fn=collate_fn)
b = next(iter(dl_tr)); print({k: tuple(v.shape) for k, v in b.items()})

## Passo 3 — Modelo, optimizer e scheduler

In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=len(classes)).to(device)
EPOCAS = 2
total_passos = EPOCAS * len(dl_tr)
# [SEU CÓDIGO AQUI] AdamW (lr=3e-5, weight_decay=0.01) e agendador linear com 10% de warmup
scaler = torch.amp.GradScaler(enabled=(device == "cuda"))

## Passo 4 — Avaliação

In [ ]:
@torch.no_grad()
def avaliar():
    modelo.eval(); preds, reais = [], []
    for lote in dl_te:
        lote = {k: v.to(device) for k, v in lote.items()}
        # [SEU CÓDIGO AQUI] forward, argmax e acúmulo em preds/reais
    modelo.train(); return np.array(preds), np.array(reais)

p, r = avaliar(); print("acurácia antes do treino:", (p == r).mean(), "(esperado ≈ 0.25)")

## Passo 5 — Loop de treino com mixed precision
Dentro de `torch.autocast`, faça o forward (o modelo já devolve `.loss` quando recebe `labels`). Depois: `scaler.scale(loss).backward()`, `scaler.unscale_`, gradient clipping, `scaler.step`, `scaler.update`, `agendador.step()`.

In [ ]:
hist, lrs, t0 = [], [], time.time()
if device == "cuda": torch.cuda.reset_peak_memory_stats()
modelo.train()
for epoca in range(EPOCAS):
    for i, lote in enumerate(dl_tr):
        lote = {k: v.to(device) for k, v in lote.items()}
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            # [SEU CÓDIGO AQUI] forward com labels -> loss
        otimizador.zero_grad()
        # [SEU CÓDIGO AQUI] backward escalado, clip, step, update, agendador
        hist.append(loss.item()); lrs.append(agendador.get_last_lr()[0])
        if i % 25 == 0: print(f"época {epoca} passo {i:4d} | loss {loss.item():.3f} | lr {lrs[-1]:.2e}")
    p, r = avaliar(); print(f"== época {epoca}: acurácia {(p == r).mean():.3f} | {time.time()-t0:.0f}s")
if device == "cuda": print(f"pico de memória de GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(hist); ax[0].set_title("loss de treino"); ax[1].plot(lrs); ax[1].set_title("taxa de aprendizado (warmup + decaimento)"); plt.show()

## Passo 6 — Avaliação com a lente da Semana 1

In [ ]:
p, r = avaliar()
# [SEU CÓDIGO AQUI] classification_report e confusion matrix

Quais duas classes o modelo mais confunde? Faz sentido?

## Passo 7 — Inferência e latência

In [ ]:
@torch.no_grad()
def classificar(textos):
    modelo.eval()
    lote = tok(textos, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
    # [SEU CÓDIGO AQUI] probabilidades por classe
    return [(classes[i], round(pr.item(), 3)) for pr, i in zip(*probs.max(-1))]

exemplos = ["Stocks fall as central bank raises interest rates", "New GPU doubles performance for AI training",
            "Striker scores twice in the cup final"]
print(classificar(exemplos))
t0 = time.perf_counter(); classificar(exemplos * 20); print(f"{(time.perf_counter()-t0)/60*1000:.1f} ms por texto (lote de 60)")

## Passo 8 — Salvar

In [ ]:
modelo.save_pretrained("classificador-agnews"); tok.save_pretrained("classificador-agnews")
print("salvo. Para recarregar: AutoModelForSequenceClassification.from_pretrained('classificador-agnews')")

## Fechamento da Fase 2
Escreva em `docs/`: em que ponto de um sistema RAG corporativo um classificador como este entraria (query routing, filtro de segurança, triagem)? Qual métrica da Semana 1 você monitoraria?